# 4 The cascade

`03` measured what the head can serve without spending a single LLM call: a properly tuned
head, per-class recall, and a `tau` splitting the thirteen issue areas into *owned* and *tail*.
This notebook builds the piece that was missing --

    embedding -> router -> { head | LLM }

-- in the order that keeps every step honest: train the router on a label that already
exists, measure what it actually buys against real controls, only then let an LLM into the
picture, and wrap the whole thing behind one `.predict()` call.

In [ ]:
import json
import os
import warnings
from pathlib import Path

import numpy as np
import torch

os.environ.setdefault("RAY_ENABLE_UV_RUN_RUNTIME_ENV", "0")
warnings.simplefilter("ignore", UserWarning)

import head
import scotus
from llm_client import CachedLLM
from vectormesh import VectorCache
from vectormesh.data import OneHot
from vectormesh.data.vectorizers import detect_device

device = detect_device()
DATA = Path("data")

profile = json.loads((DATA / "head_profile.json").read_text())
D_IN, K = profile["d_in"], profile["k"]
TAU_HEAD = profile["tau"]
OWNED = set(profile["owned"])

print(f"head: {profile['test_accuracy']:.3f} accuracy, "
      f"{profile['test_macro_recall']:.3f} macro recall ({profile['n_seeds']} seeds)")
print(f"at tau={TAU_HEAD}: owns {len(OWNED)}/{K} classes -- "
      f"{[scotus.AREAS[k] for k in sorted(OWNED)]}")

## 4.1 Reload the head -- it does not get retrained

The head in `03` is not a diagnostic throwaway; it *is* the head that ships. Retraining it
here, even to the identical config, would let a fresh roll of seed noise into a number that
`03` already spent a full section (3.5b, 3.6) getting honest. Its weights are loaded instead.

In [ ]:
testcache = VectorCache.from_hub(scotus.CACHE_REPO, split="test")
traincache = VectorCache.from_hub(scotus.CACHE_REPO, split="train")

onehot13 = OneHot(num_classes=K, label_col="label", target_col="onehot")
test_oh = testcache.map(onehot13)
train_oh = traincache.map(onehot13)

trained_head = head.build_pipeline(profile["config"], D_IN, K)
trained_head.load_state_dict(torch.load(profile["weights_path"], map_location=device))
trained_head.to(device)

head_pred, gold = head.predict(
    trained_head, head.make_loader(test_oh, profile["config"]["max_chunks"], shuffle=False),
    device)
acc = float((head_pred == gold).mean())
print(f"reloaded head: {acc:.3f} accuracy on the held-out test set "
      f"(03 reported {profile['test_accuracy']:.3f} +/- {profile['test_accuracy_std']:.3f})")

A single reload will not land exactly on the seed-averaged number above -- it is one seed, not
three -- but it should sit inside a few standard deviations of it. If it does not, something
about the saved weights or the config that rebuilds them has drifted.

## 4.2 The router's target already exists

"Is this document in a class the head serves above tau?" is answerable from the training
labels and `head_profile.json` alone. No LLM call earns this label -- it is a fact about the
*head*, not about the document's difficulty for anything else.

In [ ]:
onehot2 = OneHot(num_classes=2, label_col="owned", target_col="onehot")


def add_owned(row):
    return {"owned": int(int(row["label"]) in OWNED)}


train_router = traincache.dataset.map(add_owned).map(onehot2)
test_router = testcache.dataset.map(add_owned).map(onehot2)

train_rate = float(np.mean(train_router["owned"]))
test_rate = float(np.mean(test_router["owned"]))
print(f"owned-class share: {train_rate:.1%} train, {test_rate:.1%} test -- "
      f"a much milder imbalance than the head's own (largest/smallest class was "
      f"{max(profile['n_train'])}/{min(profile['n_train'])})")

## 4.3 Train the router

The router reuses the head's winning architecture verbatim -- same aggregator, width,
transformer depth, dropout, noise, balancing -- with `out_size=2` and the `owned` target
instead of the 13-way label. This is a real assumption, not a neutral default: a harder
13-way problem and a much easier, much less imbalanced binary one are not guaranteed to want
the same architecture. It is a defensible starting point (same embedding space, same document
lengths) rather than a settled one -- E3 asks you to check it.

Training follows `03`'s own methodology: hold out a validation slice from *training* data for
early stopping, train several seeds, and touch `test_router` exactly once, at the end.

In [ ]:
from notebooktester import param
from sklearn.metrics import roc_auc_score

split = train_router.train_test_split(test_size=0.15, seed=42)
router_train, router_valid = split["train"], split["test"]

ROUTER_EPOCHS, PATIENCE, ROUTER_SEEDS = param(60, test=1), 8, param(3, test=1)
ROUTER_CONFIG = profile["config"]

router_runs, router_models = [], []
for seed in range(ROUTER_SEEDS):
    model, pred, gold_bin = head.run(
        ROUTER_CONFIG, router_train, router_valid, ROUTER_EPOCHS, seed, D_IN, 2,
        earlystop_kwargs={"patience": PATIENCE, "save": False, "verbose": False},
        logdir="router", device=device, label_col="owned")
    router_runs.append((pred, gold_bin))
    router_models.append(model)

router_accs = [float((p == g).mean()) for p, g in router_runs]
print(f"router, validation split, {ROUTER_SEEDS} seeds: "
      f"accuracy {np.mean(router_accs):.3f} +/- {np.std(router_accs):.3f}")

In [ ]:
representative = int(np.argmin(np.abs(np.array(router_accs) - np.mean(router_accs))))
router = router_models[representative]

router_weights_path = Path("../../artefacts").resolve() / "scotus_router.pt"
torch.save(router.state_dict(), router_weights_path)
print(f"saved seed {representative} -> {router_weights_path}")

## 4.4 Did the router learn anything?

Accuracy on its own can be misleading here too -- 74% of documents are owned, so a router
that always says "owned" scores 0.74 while routing nothing. AUC and the two error rates that
actually matter for a cascade are reported alongside it: **leakage** (a tail document the
router sends to the head anyway, where it has no correct answer) and **overcaution** (an owned
document sent to the LLM for no reason, which only costs money).

In [ ]:
router_loader = head.make_loader(test_router, ROUTER_CONFIG["max_chunks"], shuffle=False)
router.eval()
scores = []
with torch.no_grad():
    for X, _ in router_loader:
        scores.append(torch.softmax(router(X.to(device)), dim=-1)[:, 1].cpu())
p_owned = torch.cat(scores).numpy()

truth = np.asarray(test_router["owned"])
router_pred = (p_owned >= 0.5).astype(int)

auc = roc_auc_score(truth, p_owned)
leakage = float((router_pred[truth == 0] == 1).mean())
overcaution = float((router_pred[truth == 1] == 0).mean())

print(f"accuracy {float((router_pred == truth).mean()):.3f}  AUC {auc:.3f}")
print(f"leakage (tail sent to the head): {leakage:.1%}")
print(f"overcaution (owned sent to the LLM): {overcaution:.1%}")

## 4.5 tau is a knob -- sweep it against real cost

`03` fixed `tau=0.6` on the head's own recall to *define* owned and tail. This is a different
`tau`, on the router's output, and it sets a real trade: raise it and fewer documents reach the
head, so accuracy on the escalated set improves but the LLM bill grows. `build_llm_predictions.py`
generated an answer for **every** test opinion for exactly this reason -- so the sweep below
costs zero additional LLM calls, whatever `tau` ends up escalating.

In [ ]:
llm = CachedLLM(DATA / "scotus_llm_predictions.json")
test_texts = scotus.opinions(testcache.dataset, doc_chars=scotus.LLM_CHARS)
llm_pred_all = np.asarray([-1 if p is None else p for p in llm.classify(test_texts, scotus.AREAS)])

head_loader = head.make_loader(test_oh, profile["config"]["max_chunks"], shuffle=False)
head_pred_all, gold_all = head.predict(trained_head, head_loader, device)

unparsed = int((llm_pred_all == -1).sum())
print(f"LLM predictions: {len(llm_pred_all)} documents, {unparsed} unparsable")
assert np.array_equal(gold_all, np.asarray(testcache.dataset["label"])), "row order drifted"

In [ ]:
import matplotlib.pyplot as plt

taus = np.linspace(0.0, 1.0, 41)
sweep_acc, sweep_cost = [], []
for tau in taus:
    owned_mask = p_owned >= tau
    pred = np.where(owned_mask, head_pred_all, llm_pred_all)
    sweep_acc.append(float((pred == gold_all).mean()))
    sweep_cost.append(float((~owned_mask).mean()))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sweep_cost, sweep_acc, marker=".")
ax.set_xlabel("fraction escalated to the LLM")
ax.set_ylabel("cascade accuracy")
ax.set_title("tau sweep: every point is the same router, a different threshold")
plt.tight_layout()
plt.show()

## 4.6 Controls -- does the router earn its keep?

A learned router is more expensive to build than two obvious alternatives: a confidence
threshold on the head's own softmax (no router at all), and escalating at random. If either
matches the router's curve, the router is not paying for itself.

In [ ]:
head_conf = []
with torch.no_grad():
    for X, _ in head_loader:
        head_conf.append(torch.softmax(trained_head(X.to(device)), dim=-1).max(-1).values.cpu())
head_conf = torch.cat(head_conf).numpy()

rng = np.random.default_rng(0)


def sweep(scores, thresholds, random_baseline=False, repeats=5):
    acc, cost = [], []
    for t in thresholds:
        if random_baseline:
            a, c = [], []
            for _ in range(repeats):
                owned_mask = rng.random(len(gold_all)) >= t
                pred = np.where(owned_mask, head_pred_all, llm_pred_all)
                a.append(float((pred == gold_all).mean()))
                c.append(float((~owned_mask).mean()))
            acc.append(float(np.mean(a)))
            cost.append(float(np.mean(c)))
        else:
            owned_mask = scores >= t
            pred = np.where(owned_mask, head_pred_all, llm_pred_all)
            acc.append(float((pred == gold_all).mean()))
            cost.append(float((~owned_mask).mean()))
    return cost, acc

conf_cost, conf_acc = sweep(head_conf, np.linspace(0.0, 1.0, 41))
rand_cost, rand_acc = sweep(None, np.linspace(0.0, 1.0, 21), random_baseline=True)
always_head_acc = float((head_pred_all == gold_all).mean())
always_llm_acc = float((llm_pred_all == gold_all).mean())

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(sweep_cost, sweep_acc, label="learned router", linewidth=2)
ax.plot(conf_cost, conf_acc, label="head confidence threshold", linestyle="--")
ax.plot(rand_cost, rand_acc, label="random escalation", linestyle=":")
ax.axhline(always_head_acc, color="grey", linewidth=1, label="always head (cost 0)")
ax.axhline(always_llm_acc, color="black", linewidth=1, label="always LLM (cost 1)")
ax.set_xlabel("fraction escalated to the LLM")
ax.set_ylabel("cascade accuracy")
ax.legend(fontsize=8)
ax.set_title("the router only earns its cost if it beats the dashed line")
plt.tight_layout()
plt.show()

Where the solid line sits above the dashed one, the router is buying something a free
confidence threshold could not: it is using information the head's own softmax does not
have, since it was trained on a target the head never saw. Where the two lines overlap, the
router is not earning its complexity at that operating point, and a confidence threshold would
have done as well for free.

## 4.7 Pick a deployment tau and wrap it up

A concrete choice: match the escalation rate the *perfect* router would have produced at the
head's own `tau` -- the `share_owned` figure `03` reported -- rather than pick a number that
happens to look good on this sweep. That keeps the LLM budget tied to a decision already
made in `03`, not to a second, unexplained knob.

In [ ]:
from cascade import Cascade

target_cost = 1 - float(np.isin(gold_all, list(OWNED)).mean())
DEPLOY_TAU = float(taus[np.argmin(np.abs(np.array(sweep_cost) - target_cost))])

cascade = Cascade(
    router=router, head=trained_head, llm=llm, areas=scotus.AREAS,
    router_max_chunks=ROUTER_CONFIG["max_chunks"],
    head_max_chunks=profile["config"]["max_chunks"],
    tau=DEPLOY_TAU, device=device,
)

cascade_pred, owned_mask = cascade.predict(test_oh, test_texts)
cascade_acc = float((cascade_pred == gold_all).mean())
cascade_cost = float((~owned_mask).mean())

print(f"deployed tau={DEPLOY_TAU:.2f}: accuracy {cascade_acc:.3f}, "
      f"{cascade_cost:.1%} escalated to the LLM")
print(f"always-head: {always_head_acc:.3f} accuracy, 0% escalated")
print(f"always-LLM:  {always_llm_acc:.3f} accuracy, 100% escalated")

`cascade.predict` reproduced the sweep's own numbers at this `tau` -- it has to, since it runs
the same router and the same lookup, just packaged as one call instead of the four separate
arrays this notebook built by hand. That packaging is the point: a caller who did not read this
notebook gets `Cascade(...).predict(dataset, texts)` and never sees the router, the padding, or
the cache key.

## Exercises

**E1 -- a label-free `Cascade.predict`.** `Collate` currently requires a `onehot` target
column to batch at all, even though `Cascade` never reads it. Write a collate function that
only needs `embed`, and use it to route a handful of opinions that have never been labelled.

**E2 -- per-class tau.** One global `tau` treats every escalation as equally costly. If an
LLM call for `Miscellaneous` is worth more (or less) than one for `Civil Rights`, how would you
fold that into the router's decision rather than the head's?

**E3 -- does the router want its own architecture?** 4.3 reused the head's winning config
outright. Run a short search over `SEARCH_SPACE` with the router's own target and see whether
the binary problem's optimum lands somewhere else, and whether it matters.

**E4 -- balance the router, not just the head.** The router's own imbalance (~74/26) is far
milder than the head's. Does `balanced=True` still help at that ratio, or does it start
trading away more accuracy than it buys back?

**E5 -- production, not cache.** Everything here reads embeddings from the published cache.
Write a `Cascade.predict_raw(texts)` that runs `Vectorizer` itself, chunk-encodes fresh
documents, and routes them -- the path a real deployment would take.